# 12 · Classificação de Alto Risco de OLA — Residual Learning

ML-4: prevê se um segmento vai entrar em **estado de alto risco** de
violação de OLA (sim/não) no horizonte D+1/D+7.

## A investigação até chegar aqui

A primeira versão treinava um classificador binário direto
(`BCEWithLogitsLoss` + `pos_weight`) e nunca bateu a baseline
(`taxa_media_movel_7d > limiar`) — nem em F1, nem em recall, em nenhuma
das 4 combinações. Testamos 12 técnicas diferentes tentando reverter
isso (Focal Loss, oversampling, arquitetura maior, transfer learning,
fine-tuning em duas etapas, stacking, mais granularidade de lag,
ensemble por `OU`) — nenhuma bateu a baseline de forma robusta e
consistente entre validação e teste.

**O que funcionou: *residual learning*.** Em vez do modelo prever o
valor absoluto da taxa de violação do zero, ele prevê **o quanto a
realidade vai desviar da baseline** — a baseline vira literalmente o
ponto de partida do cálculo, não só mais uma feature entre várias.
Isso deu **recall consistentemente maior que a baseline nas 4
combinações**, em validação e teste, com boa margem — é o resultado
mais forte de toda a investigação de ML deste projeto.

## Por que a métrica de portão mudou de F1 para recall

Pra risco de violação de SLA, um **falso negativo** (deixar passar um
risco real sem avisar) custa muito mais pra Locaweb do que um **falso
positivo** (alarme que não se confirma — a equipe checa, não acontece
nada, perde alguns minutos). F1 pesa os dois erros igual, o que não
reflete essa assimetria de custo real do negócio. Recall mede
diretamente "de todo risco real que existia, quantos a gente pegou" —
é a métrica certa quando perder caso é mais caro que alarme falso.

**O preço, sem esconder**: o número de falsos positivos sobe bastante
(às vezes 4-5x mais que a baseline). Isso é o trade-off sendo aceito
de propósito, não ignorado.

## Limiar de "alto risco"

Percentil 75 da taxa de violação **no treino** (adaptativo, evita
cravar número arbitrário).

In [ ]:
%pip install -q torch
dbutils.library.restartPython()

In [ ]:
%run ./00_config

In [ ]:
from pyspark.sql import functions as F
from datetime import timedelta
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import mlflow

mlflow.set_experiment("/Shared/antecipeai_ml_experiments")

In [ ]:
DATA_INICIO_TREINO_ML = "2024-12-01"
PERCENTIL_ALTO_RISCO = 0.75


def split_temporal_pandas(df, coluna_data="data_abertura", dias_teste=30, dias_validacao=90, dias_embargo=7):
    data_max = df[coluna_data].max()
    inicio_teste = data_max - timedelta(days=dias_teste - 1)
    fim_embargo_teste = inicio_teste - timedelta(days=dias_embargo)
    inicio_validacao = fim_embargo_teste - timedelta(days=dias_validacao - 1)
    fim_embargo_validacao = inicio_validacao - timedelta(days=dias_embargo)
    teste = df[df[coluna_data] >= inicio_teste].copy()
    validacao = df[(df[coluna_data] >= inicio_validacao) & (df[coluna_data] <= fim_embargo_teste)].copy()
    treino = df[df[coluna_data] <= fim_embargo_validacao].copy()
    return treino, validacao, teste


def metricas_classificacao(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    total_pos = tp + fn
    recall = tp / total_pos if total_pos > 0 else 0.0
    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * precisao * recall / (precisao + recall) if (precisao + recall) > 0 else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn, "recall": recall, "precisao": precisao, "f1": f1}


print("Funções prontas.")

In [ ]:
def treinar_residual_com_gate(
    nome_tabela: str, target: str, coluna_categorica: str, epocas: int = 250, lr: float = 0.002
) -> dict:
    """
    Treina um MLP como REGRESSOR do resíduo (target - taxa_media_movel_7d),
    não como classificador direto. A previsão final é
    `taxa_media_movel_7d + resíduo_previsto`, classificada como "alto risco"
    se > limiar — a baseline vira o ponto de partida embutido no cálculo,
    não uma feature entre outras. Portão decide por RECALL na validação
    (ver justificativa de negócio no cabeçalho do notebook).
    """
    colunas_lag = ["taxa_lag_1d", "taxa_lag_7d", "taxa_media_movel_7d", "taxa_media_movel_14d"]
    coluna_baseline = "taxa_media_movel_7d"

    serie = spark.table(qualified_table(SCHEMA_SILVER, nome_tabela))
    calendario = spark.table(qualified_table(SCHEMA_SILVER, "features_calendario"))
    sdf = serie.join(calendario, "data_abertura", "left").filter(F.col("data_abertura") >= DATA_INICIO_TREINO_ML)
    sdf = sdf.withColumn("is_fim_de_semana_int", F.col("is_fim_de_semana").cast("int"))

    colunas_features = colunas_lag + [
        "dia_semana_num", "trimestre", "is_feriado", "is_fim_de_semana_int",
        "qtd_kpi_regra_divergente", "qtd_duracao_suspeita",
    ]
    if coluna_categorica != "prioridade_num":
        colunas_features.append("prioridade_num")

    cols_select = ["data_abertura"] + colunas_features + [target, coluna_categorica]
    pdf_bruto = sdf.select(*cols_select).toPandas()
    pdf_bruto["data_abertura"] = pdf_bruto["data_abertura"].astype("datetime64[ns]")
    pdf_bruto[f"{coluna_categorica}_idx"] = pdf_bruto[coluna_categorica].astype("category").cat.codes
    colunas_features = colunas_features + [f"{coluna_categorica}_idx"]

    pdf = pdf_bruto.dropna(subset=colunas_lag + [target])
    treino, validacao, teste = split_temporal_pandas(pdf)
    limiar = treino[target].quantile(PERCENTIL_ALTO_RISCO)

    for df_ in [treino, validacao, teste]:
        df_["residuo"] = df_[target] - df_[coluna_baseline]

    medias = treino[colunas_features].mean()
    desvios = treino[colunas_features].std().replace(0, 1)

    def normalizar_X(df_):
        return torch.tensor(((df_[colunas_features] - medias) / desvios).fillna(0).values.astype(np.float32))

    X_treino, X_val, X_teste = normalizar_X(treino), normalizar_X(validacao), normalizar_X(teste)
    y_treino_residuo = torch.tensor(treino["residuo"].values.astype(np.float32).reshape(-1, 1))

    modelo = nn.Sequential(
        nn.Linear(len(colunas_features), 128), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1),
    )
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=1e-4)
    perda_fn = nn.HuberLoss(delta=0.5)

    for _ in range(epocas):
        modelo.train()
        otimizador.zero_grad()
        perda = perda_fn(modelo(X_treino), y_treino_residuo)
        perda.backward()
        otimizador.step()

    modelo.eval()
    with torch.no_grad():
        residuo_val = modelo(X_val).numpy().flatten()
        residuo_teste = modelo(X_teste).numpy().flatten()

    previsao_val = validacao[coluna_baseline].values + residuo_val
    previsao_teste = teste[coluna_baseline].values + residuo_teste

    def rotular(df_):
        return (df_[target] > limiar).astype(np.float32).values

    y_val, y_teste = rotular(validacao), rotular(teste)
    pred_modelo_val = (previsao_val > limiar).astype(float)
    pred_modelo_teste = (previsao_teste > limiar).astype(float)
    pred_baseline_val = (validacao[coluna_baseline].values > limiar).astype(float)
    pred_baseline_teste = (teste[coluna_baseline].values > limiar).astype(float)

    m_modelo_val = metricas_classificacao(y_val, pred_modelo_val)
    m_baseline_val = metricas_classificacao(y_val, pred_baseline_val)
    m_modelo_teste = metricas_classificacao(y_teste, pred_modelo_teste)
    m_baseline_teste = metricas_classificacao(y_teste, pred_baseline_teste)

    vencedor = "modelo" if m_modelo_val["recall"] > m_baseline_val["recall"] else "baseline"

    # Inferência real: última linha disponível por segmento (categoria + prioridade)
    pdf_bruto["data_abertura"] = pdf_bruto["data_abertura"]
    ultima_por_segmento = (
        pdf_bruto.sort_values("data_abertura")
        .groupby([coluna_categorica, "prioridade_num"], as_index=False)
        .tail(1)
        .copy()
    )
    X_ultima = normalizar_X(ultima_por_segmento)
    with torch.no_grad():
        residuo_ultima = modelo(X_ultima).numpy().flatten()
    previsao_ultima = ultima_por_segmento[coluna_baseline].values + residuo_ultima
    dias_futuro = 1 if "d1" in target else 7
    ultima_por_segmento["data_prevista"] = ultima_por_segmento["data_abertura"] + pd.Timedelta(days=dias_futuro)

    if vencedor == "modelo":
        prob_final = previsao_ultima
    else:
        prob_final = ultima_por_segmento[coluna_baseline].values

    previsoes = [
        {
            "data_prevista": row["data_prevista"],
            "alto_risco_previsto": bool(prob_final[i] > limiar),
            "probabilidade": round(float(prob_final[i]), 4),
            "produto": row[coluna_categorica] if coluna_categorica == "produto" else None,
            "grupo_designado": row[coluna_categorica] if coluna_categorica == "grupo_designado" else None,
            "prioridade_num": row["prioridade_num"],
        }
        for i, (_, row) in enumerate(ultima_por_segmento.iterrows())
    ]

    return {
        "tabela": nome_tabela, "target": target, "limiar_alto_risco": round(float(limiar), 4),
        "vencedor": vencedor,
        "recall_val_modelo": round(m_modelo_val["recall"], 4), "recall_val_baseline": round(m_baseline_val["recall"], 4),
        "f1_val_modelo": round(m_modelo_val["f1"], 4), "f1_val_baseline": round(m_baseline_val["f1"], 4),
        "recall_teste_modelo": round(m_modelo_teste["recall"], 4), "recall_teste_baseline": round(m_baseline_teste["recall"], 4),
        "fp_teste_modelo": m_modelo_teste["fp"], "fp_teste_baseline": m_baseline_teste["fp"],
        "tp_teste_modelo": m_modelo_teste["tp"], "fn_teste_modelo": m_modelo_teste["fn"],
        "total_positivos_teste": m_modelo_teste["tp"] + m_modelo_teste["fn"],
        "previsoes": previsoes,
    }


print("treinar_residual_com_gate() pronta.")

## Treinar as 4 combinações

In [ ]:
configuracoes_risco = [
    ("features_risco_ola_produto", "produto"),
    ("features_risco_ola_equipe", "grupo_designado"),
]

resultados_classificacao = {}
for nome_tabela, cat in configuracoes_risco:
    for horizonte in ["target_taxa_d1", "target_taxa_d7"]:
        chave = f"{nome_tabela}__{horizonte}"
        print(f"Treinando: {chave}")
        with mlflow.start_run(run_name=chave):
            r = treinar_residual_com_gate(nome_tabela, horizonte, cat)
            resultados_classificacao[chave] = r
            mlflow.log_params({
                "tabela": nome_tabela, "target": horizonte, "coluna_categorica": cat,
                "arquitetura": "128-64-32 (residual learning)", "lr": 0.002, "epocas": 250,
                "limiar_alto_risco": r["limiar_alto_risco"],
            })
            mlflow.log_metrics({
                "recall_val_modelo": r["recall_val_modelo"], "recall_val_baseline": r["recall_val_baseline"],
                "recall_teste_modelo": r["recall_teste_modelo"], "recall_teste_baseline": r["recall_teste_baseline"],
                "f1_val_modelo": r["f1_val_modelo"], "f1_val_baseline": r["f1_val_baseline"],
                "fp_teste_modelo": r["fp_teste_modelo"], "fp_teste_baseline": r["fp_teste_baseline"],
            })
            mlflow.set_tag("vencedor", r["vencedor"])
            mlflow.set_tag("tipo", "classificacao_residual_learning")

print(f"\n{len(resultados_classificacao)} combinações treinadas. Experimentos em: /Shared/antecipeai_ml_experiments")

## Resumo comparativo

Portão decide por **recall**, não F1 — justificativa de negócio no
cabeçalho do notebook. `fp_teste_modelo` mostra o custo aceito de
propósito (mais alarme falso, em troca de menos risco real perdido).

In [ ]:
resumo = pd.DataFrame([
    {k: v for k, v in r.items() if k != "previsoes"} for r in resultados_classificacao.values()
])
display(spark.createDataFrame(resumo))

qtd_modelo = sum(1 for r in resultados_classificacao.values() if r["vencedor"] == "modelo")
print(f"\nModelo (residual learning) venceu em {qtd_modelo} de {len(resultados_classificacao)} combinações (métrica: recall).")

## Gravar `gold.previsoes_classificacao_risco`

In [ ]:
previsoes_classificacao_pdf = pd.DataFrame([
    {
        "data_prevista": p["data_prevista"],
        "horizonte": r["target"],
        "tabela_origem": r["tabela"],
        "metodo": r["vencedor"],
        "alto_risco_previsto": p["alto_risco_previsto"],
        "probabilidade": p["probabilidade"],
        "limiar_alto_risco": r["limiar_alto_risco"],
        "produto": p["produto"],
        "grupo_designado": p["grupo_designado"],
        "prioridade_num": p["prioridade_num"],
    }
    for r in resultados_classificacao.values()
    for p in r["previsoes"]
])
previsoes_classificacao_pdf["data_geracao"] = pd.Timestamp.now()

previsoes_classificacao_sdf = spark.createDataFrame(previsoes_classificacao_pdf)
(
    previsoes_classificacao_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_GOLD, "previsoes_classificacao_risco"))
)
print(f"gold.previsoes_classificacao_risco gravada: {previsoes_classificacao_sdf.count()} linhas")
display(spark.table(qualified_table(SCHEMA_GOLD, "previsoes_classificacao_risco")))